# 🐛 Bug Hunter Agent Framework — Week 8

An agentic AI system that scrapes real buggy Python code from Stack Overflow, analyzes bugs using RAG + frontier models, and produces structured dataset entries.

## Pipeline

| Agent | Role |
|-------|------|
| **CodeScannerAgent** | Scrapes Stack Overflow for self-contained Python bug questions |
| **BugAnalysisAgent** | RAG (Chroma + MiniLM) + OpenRouter frontier model → identifies bugs |
| **BugStructureAgent** | Produces full JSON matching `buggy_dataset_nl.jsonl` schema |
| **BugReportAgent** | Generates markdown reports + saves entries to JSONL |

> **Requires**: `OPENROUTER_API_KEY` in your `.env` file. Get one free at [openrouter.ai](https://openrouter.ai/)

> **Disclaimer:** Please ensure you have the `buggy_dataset_nl.jsonl` file placed in *this same directory*. You can download it from this [Google Drive folder](https://drive.google.com/drive/folders/1AaLC3nnm6gJLU66R7mAYFuSQRkxC0WqV?usp=drive_link).
> The `bug_agents/` folder with all Python modules must also be present alongside this notebook.

## Phase 1 — Setup & Dependencies

In [1]:
%pip install -q requests beautifulsoup4 chromadb sentence-transformers pydantic gradio plotly scikit-learn numpy python-dotenv openai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import sys
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# Load environment variables
load_dotenv(override=True)

# Verify API key
api_key = os.getenv("OPENROUTER_API_KEY", "")
if api_key:
    print(f"✓ OpenRouter API key loaded (ends with ...{api_key[-4:]})")
else:
    print("⚠ OPENROUTER_API_KEY not found! Add it to your .env file.")

✓ OpenRouter API key loaded (ends with ...56ba)


## Phase 2 — Build RAG Vectorstore

Load the existing `buggy_dataset_nl.jsonl` (60 entries) into a Chroma vectorstore.
This gives the BugAnalysisAgent RAG context — similar bugs from the dataset to use as few-shot examples.

**Inspired by Week 5 Day 5**: native Chroma `PersistentClient` + `sentence-transformers/all-MiniLM-L6-v2`.

In [3]:
from bug_agents.analysis_agent import BugAnalysisAgent

# Initialize and build the vectorstore
analyzer = BugAnalysisAgent(db_path="bugs_vectorstore")
analyzer.build_vectorstore("buggy_dataset_nl.jsonl")

C:\Users\Michael Aigbovbiosa\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:35<00:00, 874kiB/s] 


### Visualize the Vectorstore

Let's see how the bug embeddings cluster by difficulty level (easy/medium/hard).

In [2]:
from chromadb import PersistentClient

# Load vectorstore data
chroma = PersistentClient(path="bugs_vectorstore")
collection = chroma.get_or_create_collection("bug_examples")
result = collection.get(include=["embeddings", "documents", "metadatas"])

vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
levels = [m.get("level", "unknown") for m in metadatas]

# Color by difficulty
color_map = {"easy": "#22c55e", "medium": "#eab308", "hard": "#ef4444", "unknown": "#6b7280"}
colors = [color_map.get(level, "#6b7280") for level in levels]

print(f"Loaded {len(vectors)} bug embeddings from vectorstore")

Loaded 60 bug embeddings from vectorstore


In [4]:
# 2D t-SNE Visualization
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(15, len(vectors) - 1))
reduced_2d = tsne_2d.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_2d[:, 0], y=reduced_2d[:, 1],
    mode="markers",
    marker=dict(size=8, color=colors, opacity=0.8),
    text=[f"Level: {level}<br>Bugs: {m.get('num_bugs', '?')}<br>{doc[:80]}..."
          for level, m, doc in zip(levels, metadatas, documents)],
    hoverinfo="text",
)])
fig.update_layout(
    title="Bug Embeddings — 2D t-SNE (colored by difficulty)",
    width=800, height=500,
    margin=dict(r=20, b=10, l=10, t=40),
)
fig.show()

In [5]:
# 3D t-SNE Visualization
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(15, len(vectors) - 1))
reduced_3d = tsne_3d.fit_transform(vectors)

fig3d = go.Figure(data=[go.Scatter3d(
    x=reduced_3d[:, 0], y=reduced_3d[:, 1], z=reduced_3d[:, 2],
    mode="markers",
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Level: {level}<br>Bugs: {m.get('num_bugs', '?')}<br>{doc[:80]}..."
          for level, m, doc in zip(levels, metadatas, documents)],
    hoverinfo="text",
)])
fig3d.update_layout(
    title="Bug Embeddings — 3D t-SNE (colored by difficulty)",
    width=900, height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)
fig3d.show()

## Phase 3 — Run the Bug Hunter Framework

Import and run the full multi-agent pipeline:

```
CodeScannerAgent → BugAnalysisAgent → BugStructureAgent → BugReportAgent
```

In [3]:
from bug_agents.bug_hunter_framework import BugHunterFramework

framework = BugHunterFramework(
    dataset_path="buggy_dataset_nl.jsonl",
    db_path="bugs_vectorstore",
    output_path="new_bugs.jsonl",
)

[Bug Hunter Framework] ============================================================
[Bug Hunter Framework] Bug Hunter Agent Framework starting up...
[Bug Hunter Framework] ============================================================
[Code Scanner Agent] Code Scanner Agent is initializing
[Code Scanner Agent] Code Scanner Agent is ready
[Bug Analysis Agent] Bug Analysis Agent is initializing
[Bug Analysis Agent] Initializing Chroma with built-in embedding...
[Bug Analysis Agent] Chroma vectorstore ready (60 entries)
[Bug Analysis Agent] Bug Analysis Agent is ready
[Bug Structure Agent] Bug Structure Agent is initializing
[Bug Structure Agent] Bug Structure Agent is ready
[Bug Report Agent] Bug Report Agent is initializing
[Bug Report Agent] Bug Report Agent is ready
[Bug Hunter Framework] ============================================================
[Bug Hunter Framework] All agents initialized. Ready to hunt bugs!
[Bug Hunter Framework] ==================================================

Backing off send_request(...) for 0.3s (requests.exceptions.ReadTimeout: HTTPSConnectionPool(host='us.i.posthog.com', port=443): Read timed out. (read timeout=15))


In [4]:
# Run the pipeline! Set use_test_data=True to skip Stack Overflow API
reports = framework.run(use_test_data=False)

[Bug Hunter Framework] 
[Bug Hunter Framework] Starting Bug Hunt...
[Bug Hunter Framework] ============================================================
[Code Scanner Agent] Scanning Stack Overflow for buggy Python code...
[Code Scanner Agent]   Searching: [python;runtime-error]
[Code Scanner Agent]   ✓ Found: How does one compare the contents of two lists of non-hashab...
[Code Scanner Agent]   Searching: [python;typeerror]
[Code Scanner Agent]   Searching: [python;nameerror]
[Code Scanner Agent]   ✓ Found: Splitting text by new line: &quot;NameError: name &#39;split...
[Code Scanner Agent]   ✓ Found: getting name error on defined variable in CMU python sandbox...
[Code Scanner Agent]   Searching: [python;indexerror]
[Code Scanner Agent] Scan complete: 3 self-contained bugs found
[Bug Hunter Framework] Analyzing 3 bugs individually...
[Bug Analysis Agent] Analyzing bug: How does one compare the contents of two lists of ...
[Bug Analysis Agent]   Calling openai/gpt-4o for analysis...
HT

In [ ]:
# View the markdown report
from IPython.display import Markdown, display

display(Markdown(framework.get_report_markdown()))

In [ ]:
# View the JSON entries
print(framework.get_entries_json())

## Phase 4 — Gradio UI

Interactive interface to scan, analyze, and view bug reports.

In [ ]:
import gradio as gr


def run_hunt(use_test_data):
    """Run the bug hunting pipeline and return results."""
    fw = BugHunterFramework(
        dataset_path="buggy_dataset_nl.jsonl",
        db_path="bugs_vectorstore",
        output_path="new_bugs.jsonl",
    )
    fw.build_vectorstore()
    reports = fw.run(use_test_data=use_test_data)
    md_report = fw.get_report_markdown()
    json_entries = fw.get_entries_json()
    return md_report, json_entries


with gr.Blocks(
    title="🐛 Bug Hunter Agent Framework",
    theme=gr.themes.Soft(),
) as demo:
    gr.Markdown("# 🐛 Bug Hunter Agent Framework")
    gr.Markdown(
        "Scans Stack Overflow for buggy Python code, analyzes bugs with RAG + frontier models, "
        "and produces structured dataset entries."
    )

    with gr.Row():
        test_mode = gr.Checkbox(
            label="Use test data (skip Stack Overflow API)",
            value=False,
        )
        run_btn = gr.Button("🔍 Start Bug Hunt", variant="primary", scale=2)

    with gr.Tabs():
        with gr.Tab("📋 Bug Reports"):
            report_output = gr.Markdown("Click 'Start Bug Hunt' to begin...")

        with gr.Tab("📦 JSON Entries"):
            gr.Markdown("Raw JSONL entries for appending to `buggy_dataset_nl.jsonl`:")
            json_output = gr.Textbox(
                label="New Entries (JSONL)",
                lines=15,
                interactive=False,
            )

    run_btn.click(
        fn=run_hunt,
        inputs=[test_mode],
        outputs=[report_output, json_output],
    )

demo.launch()